# BONSAI parser walkthrough

This notebook shows how to parse the BONSAI Balanced bundle in MARIO.
Currently, **only the hybrid supply-use table (HSUT) is supported**; HIOT and monetary SUT exports are not supported.
The checked release is `20260615`, with reference year **2016**.

## What this notebook covers

- which local files the parser expects;
- how to read a Balanced HSUT;
- how to inspect the parsed database;
- how to calculate the GHG footprint of electricity from `fc`.

## Relevant source pages

- [BONSAI project](https://bonsai.uno/)
- [BONSAI repository](https://gitlab.com/bonsamurais/bonsai)

Obtain the bundle from the project's shared data storage, under `_b Next version/Balanced`.
MARIO does not download BONSAI automatically.

## Main entry point

Use `mario.parse_bonsai(...)` with the original ZIP or its extracted release directory.

## Key arguments and local layout

- `path`: the Balanced ZIP or extracted release directory;
- `extensions`: `True` (default) includes the recalculated elementary flows; `False` reads only the economic tables.

The parser reads these files from the bundle:

```text
20260615/
├── classification.csv
├── hybrid_supply_w_characteristic_unit.csv
├── hybrid_use_w_characteristic_unit.csv
└── all_elementary_flows_AB_and_BB.csv
```

The last file is needed only when `extensions=True`. Other files in the bundle can be left in place.
See [mario.parse_bonsai](../../../api_document/mario.parse_bonsai.rst) for all arguments.

In [1]:
import mario

mario.set_log_verbosity("warning")

## Parse the HSUT

Pass the ZIP directly; no manual extraction is needed. You can also pass the extracted `20260615` directory.

The parser automatically selects `solve`, `umfpack`, `direct` and `partial` for this Python session, including subsequent calculations. Install the solver with `conda install -c conda-forge scikit-umfpack`. See the [API notes](../../../api_document/mario.parse_bonsai.rst) for details.

In [2]:
db = mario.parse_bonsai(
    path="/path/to/Balanced/20260615.zip",
)

db

/path/to/MARIO/mario/parsers/entrypoints.py:217: UserWarning: BONSAI: 362 elementary-flow rows have no classified activity/final demand and are omitted; examples: [('AT', 'A_MA_RECY'), ('BE', 'A_MA_RECY'), ('BE', 'A_SEWAGE'), ('BG', 'A_MA_LAND'), ('BG', 'A_MA_OTHE')]. Full rows are in the bonsai import report.
  matrices, indexes, units, layout = parse_bonsai_hsut(


name = BONSAI Balanced HSUT
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = from 365 to 1272 per region
Commodity = from 365 to 1335 per region
Factor of production = 1
Satellite account = 127
Consumption category = from 0 to 6 per region
Region = 50

## Inspect the parsed database

The result is a MARIO SUT database. Check its regions and the native commodity units before starting an analysis.

In [3]:
db.get_index("Region")

['AT',
 'AU',
 'BE',
 'BG',
 'BR',
 'CA',
 'CH',
 'CN',
 'CY',
 'CZ',
 'DE',
 'DK',
 'EE',
 'ES',
 'FI',
 'FR',
 'GB',
 'GR',
 'Global',
 'HR',
 'HU',
 'ID',
 'IE',
 'IN',
 'IT',
 'JP',
 'KR',
 'LT',
 'LU',
 'LV',
 'MT',
 'MX',
 'NL',
 'NO',
 'PL',
 'PT',
 'RO',
 'RU',
 'SE',
 'SI',
 'SK',
 'TR',
 'TW',
 'US',
 'WA',
 'WE',
 'WF',
 'WL',
 'WM',
 'ZA']

In [4]:
db.units["Commodity"].loc[["M_electricity"]]

,unit
M_electricity,TJ


## Electricity GHG footprint

Characterize the greenhouse gases using the built-in BONSAI profile (AR6, GWP100), then retain the aggregated GHG account. Fossil CO2, fossil and non-fossil methane, and N2O are included; biogenic CO2 stays separate. See the [GHG API](../../../api_document/mario.Database.calc_ghg.rst) for the factors.

Keeping only GHG before calculating `fc` limits the solve to one environmental row.

In [5]:
db.calc_ghg()
ghg = "GHG AR6 GWP-100"
db.keep_extensions([ghg])

db.units["Satellite account"]

,unit
Item,
GHG AR6 GWP-100,tonnes


`fc` contains direct and upstream emissions per unit of each regional commodity. For the GHG row and `M_electricity`, its units are **tonnes CO2eq/TJ**.

The four commodity coefficients affected by singular components in this release are reported as `NaN`, with a warning. The electricity markets selected below are independent of those components.

In [6]:
fc = db.fc
fc.shape

/path/to/MARIO/mario/compute/linear_solvers.py:220: PartialResultWarning: Partial linear-system result: 4 of 53400 coefficients per right-hand side are NaN because they belong to, or depend on, singular production components. Singular components: [{'nodes': [('TW', 'Commodity', 'C_Beany'), ('TW', 'Commodity', 'M_Beany')], 'count': 2}, {'nodes': [('TW', 'Commodity', 'C_Eggs_ggs'), ('TW', 'Commodity', 'M_Eggs_ggs')], 'count': 2}]. Only independent equations are solved; the full system may be inconsistent or have nonunique solutions. Source matrices are unchanged. Totals including undefined coefficients must propagate NaN (use skipna=False for pandas reductions).
  solved = np.asarray(factor.solve(rhs_values), dtype=np.float64)


(1, 53400)

Select `M_electricity` across regions. Each result is the GHG footprint of **1 kWh supplied by that regional electricity market**, including its upstream supply chain.

**Unit conversion:** multiplying by **3.6** converts tonnes CO2eq/TJ into g CO2eq/kWh, since 1 tonne = 10⁶ g and 1 kWh = 3.6 × 10⁻⁶ TJ.

In [7]:
electricity_ghg = fc.loc[
    ghg, (slice(None), slice(None), "M_electricity")
] * 3.6

electricity_ghg.droplevel([1, 2]).to_frame("g CO2eq/kWh")

,g CO2eq/kWh
Region,
AT,244.583914
AU,735.553038
BE,233.659069
BG,545.943557
BR,123.240659
CA,155.694559
CH,177.398858
CN,859.220591
CY,1745.262889


## Notes

- Activity and commodity classifications can differ between regions. This is a [special case of the index API](../../../get_started/nomenclature.rst); see [Basic inspections](../../../user_guide/inspection/basic_inspections.ipynb) for the inspection workflow.
- Value-added accounts are not available in these inputs; MARIO uses zero placeholders.
- Satellite labels use `name | compartment`; IDs disambiguate homonyms only. Units and native identities remain in `db.units["Satellite account"]` and `db.info["bonsai"]["satellites"]`. `db.calc_ghg()` auto-detects the BONSAI profile; see the [GHG API](../../../api_document/mario.Database.calc_ghg.rst) for its factors and scope.
- The default `calc_all=False` keeps the imported blocks sparse. Full tables are large: reduce the model before requesting full inverse matrices.
- Import diagnostics are available in `db.info["bonsai"]`. Unmatched elementary-flow rows generate a warning by default; use `unmatched_extensions="raise"` to reject them.